In [2]:
from datasets import load_dataset

dataset = load_dataset("khalidalt/SANAD", "default", split="train")


C:\Users\saleen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(dataset)

Dataset({
    features: ['article', 'category'],
    num_rows: 99810
})


In [4]:
dataset[0]

{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n',
 'category': 'Politics'}

In [5]:
import re

In [6]:
def clean_text(text):
    text = re.sub(r'[إأآا]', 'ا', text)
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    # Remove tatweel
    text = re.sub(r'\u0640', '', text)
    # Remove non-Arabic characters except spaces and punctuation
    text = re.sub(r'[^\u0600-\u06FF\s\.\،\؟\!]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
sample = dataset[0]['article']
sample =sample[:200]
cleaned_sample = clean_text(sample)
after_cleaning = cleaned_sample
before_cleaning = sample

In [8]:
print("Before cleaning:")
print(before_cleaning)
print("\nAfter cleaning:")
print(after_cleaning)

Before cleaning:

المخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف

الحدث.نت

التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي

After cleaning:
المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي


In [9]:
# ! pip install qdrant-client langchain langchain-community langchain-qdrant sentence-transformers transformers datasets arabert fastembed -q

In [10]:
# !pip install langchain-text-splitters -q

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '،', '.', ' ']
)

C:\Users\saleen\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [12]:
def prepare_documents(dataset,max_articles=5000):
    docs=[]
    for i, item in enumerate(dataset):
        if i >= max_articles:
            break
        text=clean_text(item['article'])
        if len(text) < 50:
            continue
        chunks = text_splitter.split_text(text)
        for j,chunk in enumerate(chunks):
            docs.append({
                'id': f"{[i]}_{j}",
                'text': chunk,
                'category':item.get('category', 'unknown'),
                'source_idx': i
            })
    return docs
docs = prepare_documents(dataset)
print(f"total chunks: {len(docs)}")
print(docs[0])

total chunks: 30148
{'id': '[0]_0', 'text': 'المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يراس الوفد الحكومي للمفاوضات اليمنية اجندة وجدول اعمال مشاورات جنيف', 'category': 'Politics', 'source_idx': 0}


In [13]:
print(dataset.column_names)
print(dataset[0])

['article', 'category']
{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n', 'category': 'Politics'}


In [14]:
# pip install qdrant-client -q

In [15]:
! pip install langchain-community -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\saleen\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [16]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, SparseVectorParams, SparseIndexParams)
from langchain_community.embeddings import HuggingFaceEmbeddings

In [17]:
client = QdrantClient(path="./qdrant_db")

In [18]:
#! pip install sentence-transformers

In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

C:\Users\saleen\AppData\Local\Temp\ipykernel_18456\4233599945.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 157.41it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform

In [20]:
test = embeddings.embed_query("ما هو اسمك؟")
print(test)

[0.0023290105164051056, 0.04413722828030586, 0.018427064642310143, -0.026165178045630455, 0.043646860867738724, 0.029785512015223503, -0.0511583611369133, 0.04085143283009529, -0.019091885536909103, 0.02737371064722538, 0.015309656038880348, 0.006134508177638054, -0.01073518767952919, -0.01511376816779375, 0.02527894824743271, 0.030698757618665695, 0.034358710050582886, -0.030433766543865204, 0.002242975402623415, -0.03646301105618477, 0.047753237187862396, -0.020411686971783638, 0.028418492525815964, -0.05337229371070862, -0.025023112073540688, -0.0141666941344738, 0.0481475405395031, 0.034374915063381195, 0.001132449135184288, -0.050957825034856796, -0.03912819176912308, 0.009939317591488361, 0.03823384642601013, -0.008800783194601536, -0.015041724778711796, -0.00019214293570257723, 0.02376093715429306, 0.0013478135224431753, 0.018861981108784676, 0.021353181451559067, 0.024402154609560966, -0.007240560371428728, -0.02356981486082077, 0.03870289400219917, -0.052001018077135086, -0.05

In [21]:
collection_name = "arabic_news"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    ),
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)
print("Collection created successfully.", collection_name)

Collection created successfully. arabic_news


In [23]:
print(client.get_collection(collection_name))

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors={'sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=False, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=No

In [32]:
!pip install fastembed==0.4.2


ERROR: Ignored the following versions that require a different python version: 0.0.1 Requires-Python >=3.8.0,<3.12; 0.0.1a1 Requires-Python >=3.8.0,<3.12; 0.0.1a2 Requires-Python >=3.8.0,<3.12; 0.0.1a3 Requires-Python >=3.8.0,<3.12; 0.0.2 Requires-Python >=3.8.0,<3.12; 0.0.2a0 Requires-Python >=3.8.0,<3.12; 0.0.3 Requires-Python >=3.8.0,<3.12; 0.0.3a1 Requires-Python >=3.8.0,<3.12; 0.0.4 Requires-Python >=3.8.0,<3.12; 0.0.5 Requires-Python >=3.8.0,<3.12; 0.0.5a1 Requires-Python >=3.8.0,<3.12; 0.0.5a2 Requires-Python >=3.8.0,<3.12; 0.1.0 Requires-Python >=3.8.0,<3.12; 0.1.1 Requires-Python >=3.8.0,<3.12; 0.1.2 Requires-Python >=3.8.0,<3.12; 0.1.3 Requires-Python >=3.8.0,<3.12; 0.2.0 Requires-Python >=3.8.0,<3.13; 0.2.1 Requires-Python >=3.8.0,<3.13; 0.2.2 Requires-Python >=3.8.0,<3.13; 0.2.3 Requires-Python >=3.8.0,<3.13; 0.2.4 Requires-Python >=3.8.0,<3.13; 0.2.5 Requires-Python >=3.8.0,<3.13; 0.2.6 Requires-Python >=3.8.0,<3.13; 0.2.7 Requires-Python >=3.8.0,<3.13; 0.3.0 Requires-Pyth

In [ ]:
from qdrant_client.models import PointStruct , SparseVector
from fastembed import SparseTextEmbedding
from langchain_community.embedings import HuggingFaceEmbeddings
import uuid

ModuleNotFoundError: No module named 'fastembed'

In [34]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 125.02it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

NameError: name 'SparseTextEmbedding' is not defined